# ScanOps V16-7B 어댑터 → GGUF (병합 없이, Ollama ADAPTER 방식)

메모리 결정: **병합 금지.** `convert_lora_to_gguf.py`로 LoRA 어댑터만 GGUF화한다.
산출 `v16_7b_lora.gguf`(~40MB)를 로컬에 받아 Ollama ADAPTER로 배포한다.

## ① 의존성 + llama.cpp

In [ ]:
!pip uninstall -y -q tensorflow tensorflow-cpu tf-keras torchao 2>/dev/null
!pip -q install -U transformers peft accelerate sentencepiece gguf
!rm -rf llama.cpp && git clone -q https://github.com/ggml-org/llama.cpp
print('① 준비 완료')

## ② adapter_v16_7b.zip 업로드

In [ ]:
import os, shutil, zipfile
if os.path.exists('adapter7b'): shutil.rmtree('adapter7b')
os.makedirs('adapter7b', exist_ok=True)
from google.colab import files
up = files.upload()   # adapter_v16_7b.zip 선택
z = [k for k in up if k.endswith('.zip')][0]
with zipfile.ZipFile(z) as f: f.extractall('adapter7b')
print('adapter ok:', os.path.exists('adapter7b/adapter_model.safetensors'))

## ③ 어댑터만 GGUF 변환 (convert_lora_to_gguf.py — 병합 안 함)

In [ ]:
# 어댑터만 GGUF로 변환 (베이스 가중치 다운로드 X — config만 HF에서 받음)
# 주의: --base 는 '로컬 디렉토리'용. HF 모델 ID는 반드시 --base-model-id 로 넘긴다.
!python llama.cpp/convert_lora_to_gguf.py adapter7b \
    --base-model-id Qwen/Qwen2.5-Coder-7B-Instruct \
    --outfile v16_7b_lora.gguf --outtype f16
import os
assert os.path.exists('v16_7b_lora.gguf'), '변환 실패 — 위 로그 확인'
print('✅ GGUF 어댑터 크기(MB):', round(os.path.getsize('v16_7b_lora.gguf')/1e6, 1))


## ④ 다운로드 → 로컬 models/ 로 이동

In [ ]:
from google.colab import files
files.download('v16_7b_lora.gguf')